# Weight Documentation Timing vs Med Administration

Analyzes how many patients have meds administered before their first weight recording,
and what med dose units are used in those cases.

In [1]:
import pandas as pd
import duckdb

med_cont_df = pd.read_parquet("../output/intermediate/meds_cont_df.parquet")
weight_df = pd.read_parquet("../output/intermediate/weight_lookup.parquet")

print(f"Meds: {med_cont_df.shape[0]:,} rows, {med_cont_df['hospitalization_id'].nunique():,} patients")
print(f"Weights: {weight_df.shape[0]:,} rows, {weight_df['hospitalization_id'].nunique():,} patients")

Meds: 299,829 rows, 1,575 patients
Weights: 13,277 rows, 1,614 patients


In [2]:
# First weight recorded per patient
first_weight = duckdb.sql("""
    SELECT hospitalization_id,
           MIN(recorded_dttm) AS first_weight_dttm
    FROM weight_df
    GROUP BY hospitalization_id
""").fetchdf()

# Join first weight time onto meds
meds = med_cont_df.merge(first_weight, on="hospitalization_id", how="left")
meds["has_prior_weight"] = meds["first_weight_dttm"].notna() & (meds["first_weight_dttm"] <= meds["admin_dttm"])
meds["has_any_weight"] = meds["first_weight_dttm"].notna()

print(f"Total med rows: {len(meds):,}")
print(f"Rows with prior weight: {meds['has_prior_weight'].sum():,}")
print(f"Rows WITHOUT prior weight: {(~meds['has_prior_weight']).sum():,}")

Total med rows: 299,829
Rows with prior weight: 296,764
Rows WITHOUT prior weight: 3,065


In [3]:
# Patient-level grouping
total_patients = meds["hospitalization_id"].nunique()
no_prior = meds[~meds["has_prior_weight"]]
has_prior = meds[meds["has_prior_weight"]]

patients_always_prior = set(has_prior["hospitalization_id"].unique()) - set(no_prior["hospitalization_id"].unique())
patients_mixed = set(no_prior["hospitalization_id"].unique()) & set(has_prior["hospitalization_id"].unique())
patients_never_prior = set(no_prior["hospitalization_id"].unique()) - set(has_prior["hospitalization_id"].unique())
patients_no_weight_ever = set(meds["hospitalization_id"].unique()) - set(weight_df["hospitalization_id"].unique())

summary = pd.DataFrame([
    {"Group": "Always had weight before meds", "Patients": len(patients_always_prior), "Pct": len(patients_always_prior) / total_patients * 100},
    {"Group": "Some meds before first weight", "Patients": len(patients_mixed), "Pct": len(patients_mixed) / total_patients * 100},
    {"Group": "Never had weight before any med", "Patients": len(patients_never_prior), "Pct": len(patients_never_prior) / total_patients * 100},
    {"Group": "  └─ Weight recorded later", "Patients": len(patients_never_prior - patients_no_weight_ever), "Pct": len(patients_never_prior - patients_no_weight_ever) / total_patients * 100},
    {"Group": "  └─ No weight EVER recorded", "Patients": len(patients_no_weight_ever), "Pct": len(patients_no_weight_ever) / total_patients * 100},
], columns=["Group", "Patients", "Pct"])
summary["Pct"] = summary["Pct"].map("{:.1f}%".format)

print(f"Total patients with continuous meds: {total_patients:,}")
summary

Total patients with continuous meds: 1,575


,Group,Patients,Pct
0,Always had weight before meds,1253,79.6%
1,Some meds before first weight,312,19.8%
2,Never had weight before any med,10,0.6%
3,└─ Weight recorded later,4,0.3%
4,└─ No weight EVER recorded,6,0.4%


In [4]:
# Med dose unit distribution — all rows without prior weight
no_prior_filtered = no_prior[no_prior["has_any_weight"]]  # exclude never-weight patients

unit_dist = (
    no_prior_filtered
    .groupby("med_dose_unit")
    .agg(rows=("hospitalization_id", "size"), patients=("hospitalization_id", "nunique"))
    .sort_values("rows", ascending=False)
    .reset_index()
)
unit_dist["row_pct"] = (unit_dist["rows"] / unit_dist["rows"].sum() * 100).map("{:.1f}%".format)

print(f"Unique dose units in pre-weight rows: {unit_dist.shape[0]}")
unit_dist

Unique dose units in pre-weight rows: 5


,med_dose_unit,rows,patients,row_pct
0,mcg/kg/min,2368,311,83.3%
1,Units/min,292,39,10.3%
2,mcg/kg/hr,138,11,4.9%
3,mg/hr,34,4,1.2%
4,ng/kg/min,10,3,0.4%


In [5]:
# Med category × dose unit breakdown for pre-weight rows
cat_unit = (
    no_prior_filtered
    .groupby(["med_category", "med_dose_unit"])
    .agg(rows=("hospitalization_id", "size"), patients=("hospitalization_id", "nunique"))
    .sort_values("rows", ascending=False)
    .reset_index()
)
cat_unit

,med_category,med_dose_unit,rows,patients
0,norepinephrine,mcg/kg/min,1010,232
1,propofol,mcg/kg/min,651,92
2,vasopressin,Units/min,292,39
3,dobutamine,mcg/kg/min,185,17
4,epinephrine,mcg/kg/min,178,72
5,phenylephrine,mcg/kg/min,154,17
6,dexmedetomidine,mcg/kg/hr,138,11
7,dopamine,mcg/kg/min,102,14
8,milrinone,mcg/kg/min,75,4
9,midazolam,mg/hr,34,4


In [6]:
# Time gap: med admin → first weight recording (for patients who eventually got weight)
no_prior_filtered = no_prior_filtered.copy()
no_prior_filtered["gap_hours"] = (
    (no_prior_filtered["first_weight_dttm"] - no_prior_filtered["admin_dttm"])
    .dt.total_seconds() / 3600
)

gap_stats = no_prior_filtered["gap_hours"].describe(percentiles=[0.25, 0.5, 0.75]).to_frame("value")
gap_stats["value"] = gap_stats["value"].map("{:.1f}".format)
print("Time gap (hours) from med admin to first weight recording:")
gap_stats

Time gap (hours) from med admin to first weight recording:


,value
count,2842.0
mean,17.4
std,18.7
min,0.0
25%,2.2
50%,9.9
75%,29.0
max,71.4


In [7]:
# Time gap by med category
gap_by_cat = (
    no_prior_filtered
    .groupby("med_category")["gap_hours"]
    .agg(["median", "mean", "count"])
    .sort_values("count", ascending=False)
    .rename(columns={"median": "median_hours", "mean": "mean_hours", "count": "rows"})
)
gap_by_cat["median_hours"] = gap_by_cat["median_hours"].map("{:.1f}".format)
gap_by_cat["mean_hours"] = gap_by_cat["mean_hours"].map("{:.1f}".format)
gap_by_cat

,median_hours,mean_hours,rows
med_category,,,
norepinephrine,7.0,14.9,1010
propofol,9.8,17.5,651
vasopressin,13.0,19.4,292
dobutamine,13.0,21.8,185
epinephrine,2.0,8.7,178
phenylephrine,21.5,25.4,154
dexmedetomidine,22.7,22.5,138
dopamine,6.7,11.3,102
milrinone,39.3,37.5,75


In [8]:
# Compare weight-based unit usage: pre-weight vs post-weight rows
import re

def is_weight_based(unit):
    if pd.isna(unit):
        return False
    return bool(re.search(r"/kg[/\s]", str(unit).lower()) or str(unit).lower().endswith("/kg"))

meds["weight_based"] = meds["med_dose_unit"].apply(is_weight_based)

comparison = pd.DataFrame([
    {
        "Group": "No prior weight",
        "Weight-based rows": int(no_prior["med_dose_unit"].apply(is_weight_based).sum()),
        "Total rows": len(no_prior),
        "Patients": no_prior["hospitalization_id"].nunique(),
    },
    {
        "Group": "Has prior weight",
        "Weight-based rows": int(has_prior["med_dose_unit"].apply(is_weight_based).sum()),
        "Total rows": len(has_prior),
        "Patients": has_prior["hospitalization_id"].nunique(),
    },
])
comparison["Pct weight-based"] = (comparison["Weight-based rows"] / comparison["Total rows"] * 100).map("{:.1f}%".format)
comparison

,Group,Weight-based rows,Total rows,Patients,Pct weight-based
0,No prior weight,2707,3065,322,88.3%
1,Has prior weight,257837,296764,1565,86.9%
